<h1 dir="rtl" style="text-align: right;">
تحلیل اکتشافی داده های اضطراب اجتماعی
</h1>

<p dir="rtl" style="text-align: right;">
<strong>فاز دوم: <span dir="ltr">EDA</span></strong>
</p>

<p dir="rtl" style="text-align: right;">
اعضای تیم: علی خوش اخلاق، محمدحسین میرمعصومی، آرمین نورمحمدی، علی کریمی، محسن منصف
</p>

<h2 dir="rtl" style="text-align: right;">
1. کتابخانه ها و تنظیمات
</h2>

<p dir="rtl" style="text-align: right;">
از پایتون 3.13 استفاده کنید و ورژن های زیر
</p>

In [ ]:
# pandas==3.0.5 numpy==2.2.6 plotly==7.1.0 scipy==1.16.3 statsmodels==0.15.0 matplotlib==3.11.2

In [ ]:
import numpy as np
import pandas as pd
import plotly
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import matplotlib.pyplot as plt
import seaborn as sns
import scipy
import statsmodels
from plotly.subplots import make_subplots
from scipy import stats
from statsmodels.stats.proportion import proportion_confint

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pio.renderers.default = "notebook"
px.defaults.template = "plotly_white"
px.defaults.height = 450
BLUE = "#4C78A8"
RED = "#E45756"
TARGET = "Anxiety Level (1-10)"
LABEL = "Target"

<h2 dir="rtl" style="text-align: right;">
2. داده های اولیه
</h2>

In [ ]:
df = pd.read_csv("social_anxiety_dataset.csv")
print(df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe(include="all").T

In [ ]:
pd.DataFrame({
    'Missing': df.isnull().sum(),
    'Missing_%': (df.isna().mean() * 100).round(1),
    'unique': df.nunique(),
})

In [ ]:
print(df.duplicated().sum())

<h2 dir="rtl" style="text-align: right;">
مشاهدات اولیه
</h2>

<p dir="rtl" style="text-align: right;">
• سطر هامون 2030 تاس و ستون هامون 22 تاس
</p>

<p dir="rtl" style="text-align: right;">
• 9 تا ستون با داده گمشده داریم
</p>

<p dir="rtl" style="text-align: right;">
&nbsp;&nbsp;&nbsp;&nbsp;◦ که <span dir="ltr">Therapy History</span> با 90% داده گمشده بدترینه
</p>

<p dir="rtl" style="text-align: right;">
• <span dir="ltr">Sleep Hours</span>، <span dir="ltr">Physical Activity</span> و <span dir="ltr">Alcohol</span> مقدار منفی دارند
</p>

<p dir="rtl" style="text-align: right;">
• <span dir="ltr">Stress Level</span> مقدار 15 دارد که منطقی نیست
</p>

<p dir="rtl" style="text-align: right;">
• دوتا ستون <span dir="ltr">Target</span> و <span dir="ltr">is_Anxious</span> تغریبا یکسانند
</p>

<p dir="rtl" style="text-align: right;">
• ستون <span dir="ltr">Heart Rate</span> و <span dir="ltr">Caffeine</span> مقدار خارج از بازه دارند
</p>

<h2 dir="rtl" style="text-align: right;">
3. تحلیل و پاک سازی داده
</h2>

<h3 dir="rtl" style="text-align: right;">
3.1 جمعیت شناختی:
<span dir="ltr">Age, Gender, Occupation</span>
</h3>

<h3 dir="rtl" style="text-align: right;">
3.2 سبک زندگی:
<span dir="ltr">Sleep Hours, Physical Activity, Caffeine Intake, Alcohol Consumption, Smoking, Diet Quality</span>
</h3>

In [ ]:
#==================
# Numeric Data
#==================
Sleep_Hours = df['Sleep Hours'].copy()
Physical_Activity = df['Physical Activity (hrs/week)'].copy()
Caffeine_Intake = df['Caffeine Intake (mg/day)'].copy()
Alcohol_Consumption = df['Alcohol Consumption (drinks/week)'].copy()


# =========================
# Categorical Data
# =========================
Smoking = df['Smoking'].copy()
Diet_Quality = df['Diet Quality (1-10)'].copy()

In [ ]:
# =========================
# Plot Numerical Data 
# =========================
fig, axes = plt.subplots(4, 1, figsize=(12, 22))

# --- 1. Sleep Hours ---
axes[0].hist(
    Sleep_Hours,
    bins=20,
    color='cornflowerblue',
    edgecolor='black',
    alpha=0.8
)
axes[0].set_title('Distribution of Sleep Hours', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Sleep Hours', fontsize=11)
axes[0].set_ylabel('Frequency', fontsize=11)
axes[0].grid(axis='y', alpha=0.3)


# --- 2. Physical Activity ---
axes[1].hist(
    Physical_Activity,
    bins=20,
    color='cornflowerblue',
    edgecolor='black',
    alpha=0.8
)
axes[1].set_title('Distribution of Physical Activity', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Physical Activity (hrs/week)', fontsize=11)
axes[1].set_ylabel('Frequency', fontsize=11)
axes[1].grid(axis='y', alpha=0.3)


# --- 3. Caffeine Intake ---
axes[2].hist(
    Caffeine_Intake,
    bins=20,
    color='cornflowerblue',
    edgecolor='black',
    alpha=0.8
)
axes[2].set_title('Distribution of Caffeine Intake', fontsize=14, fontweight='bold')
axes[2].set_xlabel('Caffeine Intake (mg/day)', fontsize=11)
axes[2].set_ylabel('Frequency', fontsize=11)
axes[2].grid(axis='y', alpha=0.3)


# --- 4. Alcohol Consumption ---
axes[3].hist(
    Alcohol_Consumption,
    bins=10,
    color='cornflowerblue',
    edgecolor='black',
    alpha=0.8
)
axes[3].set_title('Distribution of Alcohol Consumption', fontsize=14, fontweight='bold')
axes[3].set_xlabel('Alcohol Consumption (drinks/week)', fontsize=11)
axes[3].set_ylabel('Frequency', fontsize=11)
axes[3].grid(axis='y', alpha=0.3)

# =========================
# Layout
# =========================
fig.suptitle(
    'Distribution of Numeric Variables',
    fontsize=18,
    fontweight='bold',
    y=0.995
)

plt.tight_layout(rect=[0, 0, 1, 0.98])
plt.show()

In [ ]:
# =========================
# Plot categorical Data
# =========================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- 1.Smoking ---
smoking_counts = Smoking.value_counts()

axes[0].bar(
    smoking_counts.index.astype(str),
    smoking_counts.values,
    edgecolor='black',
    alpha=0.8
)

axes[0].set_title('Smoking Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Smoking', fontsize=11)
axes[0].set_ylabel('Frequency', fontsize=11)
axes[0].grid(axis='y', alpha=0.3)


# --- 2.Diet Quality ---
diet_counts = Diet_Quality.value_counts().sort_index()

axes[1].bar(
    diet_counts.index.astype(str),
    diet_counts.values,
    edgecolor='black',
    alpha=0.8
)

axes[1].set_title('Diet Quality Distribution', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Diet Quality (1-10)', fontsize=11)
axes[1].set_ylabel('Frequency', fontsize=11)
axes[1].grid(axis='y', alpha=0.3)


# =========================
# Layout
# =========================
fig.suptitle(
    'Distribution of Categorical and Discrete Variables',
    fontsize=17,
    fontweight='bold'
)

plt.tight_layout()
plt.show()

In [ ]:
# ========================================
# remove invalid data from Numeric columns 
# ========================================

df_clean = df.copy()

df_clean.loc[df_clean['Sleep Hours'] < 0, 'Sleep Hours'] = np.nan

df_clean.loc[df_clean['Physical Activity (hrs/week)'] < 0,
             'Physical Activity (hrs/week)'] = np.nan

df_clean.loc[df_clean['Alcohol Consumption (drinks/week)'] < 0,
             'Alcohol Consumption (drinks/week)'] = np.nan

In [ ]:
# =========================
# Numeric columns 
# =========================

numeric_cols = [
    'Age',
    'Sleep Hours',
    'Physical Activity (hrs/week)',
    'Caffeine Intake (mg/day)',
    'Alcohol Consumption (drinks/week)'
]
# ===========================================
# Calculate median and replace missing values
# ===========================================

for col in numeric_cols:
    median_value = df_clean[col].median()
    df_clean[col] = df_clean[col].fillna(median_value)
    
# محاسبه صدک ۹۹
upper_limit = df_clean['Caffeine Intake (mg/day)'].quantile(0.98)
print("Upper limit (99th percentile):", upper_limit)

# سقف‌گذاری
df_clean.loc[df_clean['Caffeine Intake (mg/day)'] > upper_limit, 
             'Caffeine Intake (mg/day)'] = upper_limit

In [ ]:
# ==============================================================
# Calculate mode and replace missing values for Categorical Data
# ==============================================================
smoking_mode = Smoking.mode()[0]
print('Smoking Mode:', smoking_mode)
df_clean['Smoking'] = df_clean['Smoking'].fillna(smoking_mode)

Diet_Quality_mode = Diet_Quality.mode()[0]
print('Diet Quality:', Diet_Quality_mode)
df_clean['Diet Quality (1-10)'] = df_clean['Diet Quality (1-10)'].fillna(Diet_Quality_mode)

<h3 dir="rtl" style="text-align: right;">
3.3 فیزیولوژیک:
<span dir="ltr">Heart Rate, Breathing Rate, Sweating Level, Dizziness</span>
</h3>

<h4 dir="rtl" style="text-align: right;">
انتخاب متغیرهای فیزیولوژیک
</h4>

<p dir="rtl" style="text-align: right;">
در این بخش، چهار متغیر فیزیولوژیک شامل
<span dir="ltr">Heart Rate</span>،
<span dir="ltr">Breathing Rate</span>،
<span dir="ltr">Sweating Level</span>
و
<span dir="ltr">Dizziness</span>
از دیتافریم اصلی انتخاب شدند.
</p>

<p dir="rtl" style="text-align: right;">
برای جلوگیری از اعمال تغییرات ناخواسته روی دیتافریم اصلی، با استفاده از متد
<span dir="ltr">copy()</span>
یک کپی مستقل از ستون‌های انتخاب‌شده در دیتافریم
<span dir="ltr">physiological_df</span>
ذخیره شد.
</p>

In [ ]:
physiological_df = df[
    [
        "Heart Rate (bpm)",
        "Breathing Rate (breaths/min)",
        "Sweating Level (1-5)",
        "Dizziness"
    ]
].copy()

<h4 dir="rtl" style="text-align: right;">
بررسی نوع داده و مقادیر نامعتبر
</h4>

<p dir="rtl" style="text-align: right;">
سه ستون اول دیتافریم
<span dir="ltr">physiological_df</span>
باید دارای مقادیر عددی باشند. به همین دلیل، مقادیر این ستون‌ها با استفاده از تابع
<span dir="ltr">pd.to_numeric()</span>
به نوع عددی تبدیل شدند.
</p>

<p dir="rtl" style="text-align: right;">
با تنظیم پارامتر
<span dir="ltr">errors="coerce"</span>،
هر مقدار غیرقابل‌تبدیل به عدد به
<span dir="ltr">NaN</span>
تبدیل می‌شود. سپس تعداد مقادیر گمشده یا غیرعددی هر ستون بررسی شد.
</p>

<p dir="rtl" style="text-align: right;">
در پایان نیز نوع داده تمام ستون‌های فیزیولوژیک نمایش داده شد. نتایج نشان داد که سه ستون اول با موفقیت به نوع عددی تبدیل شده‌اند و هیچ مقدار گمشده یا غیرعددی در آن‌ها وجود ندارد.
</p>

In [ ]:
for col in physiological_df.columns[:3]:

    physiological_df[col] = pd.to_numeric(physiological_df[col], errors="coerce")

    print(col, "=>", physiological_df[col].isna().sum(), "non-numeric or missing values")

print(physiological_df.dtypes)

<h4 dir="rtl" style="text-align: right;">
بررسی اولیه مقادیر ستون‌های عددی
</h4>

<p dir="rtl" style="text-align: right;">
برای بررسی اولیه سه متغیر عددی فیزیولوژیک، از متد
<span dir="ltr">describe()</span>
استفاده شد. این متد آماره‌هایی مانند تعداد داده‌ها، میانگین، انحراف معیار، کمینه، چارک‌ها و بیشینه را نمایش می‌دهد.
</p>

<p dir="rtl" style="text-align: right;">
با بررسی مقدار کمینه
<span dir="ltr">(min)</span>
مشخص شد که هیچ مقدار صفر یا منفی در این سه ستون وجود ندارد. با این حال، مقدار بیشینه ستون
<span dir="ltr">Heart Rate</span>
نسبت به سایر مقادیر غیرعادی به نظر می‌رسد و لازم است در مرحله بعد با دقت بیشتری بررسی شود.
</p>

In [ ]:
physiological_df[
    [
        "Heart Rate (bpm)",
        "Breathing Rate (breaths/min)",
        "Sweating Level (1-5)"
    ]
].describe()

<h4 dir="rtl" style="text-align: right;">
بررسی مقادیر بالای ضربان قلب
</h4>

<p dir="rtl" style="text-align: right;">
در آمار توصیفی، مقدار بیشینه ستون
<span dir="ltr">Heart Rate</span>
نسبت به سایر مقادیر غیرعادی به نظر می‌رسید. برای بررسی دقیق‌تر، فراوانی مقادیر این ستون محاسبه و مقادیر به‌صورت صعودی مرتب شدند.
</p>

<p dir="rtl" style="text-align: right;">
با نمایش ده مقدار انتهایی مشخص شد که مقدار
<span dir="ltr">220 bpm</span>
تعداد ۳۰ بار تکرار شده و فاصله قابل‌توجهی با سایر مقادیر دارد. بنابراین، این مقدار به‌عنوان یک مقدار مشکوک در نظر گرفته شد و در مرحله بعد با روش
<span dir="ltr">IQR</span>
بررسی می‌شود.
</p>

In [ ]:
physiological_df["Heart Rate (bpm)"].value_counts().sort_index().tail(10)

<h4 dir="rtl" style="text-align: right;">
شناسایی داده‌های پرت ضربان قلب
</h4>

<p dir="rtl" style="text-align: right;">
به دلیل وجود مقدار بیشینه غیرعادی، برای شناسایی داده‌های پرت ستون
<span dir="ltr">Heart Rate</span>
از روش
<span dir="ltr">IQR</span>
استفاده شد. این روش بر اساس چارک اول و سوم محاسبه می‌شود و در مقایسه با روش‌هایی مانند
<span dir="ltr">Z-score</span>
حساسیت کمتری نسبت به مقادیر بسیار بزرگ دارد.
</p>

<p dir="rtl" style="text-align: right;">
ابتدا چارک اول
<span dir="ltr">(Q1)</span>
و چارک سوم
<span dir="ltr">(Q3)</span>
محاسبه شدند. سپس فاصله میان‌چارکی از تفاضل این دو مقدار به دست آمد و کران‌های پایین و بالا با ضریب ۱٫۵ تعیین شدند.
</p>

<p dir="rtl" style="text-align: right;">
نتایج نشان داد که کران پایین برابر با
<span dir="ltr">29.5</span>
و کران بالا برابر با
<span dir="ltr">153.5</span>
است. در مجموع ۳۰ داده پرت شناسایی شد که تمام آن‌ها دارای مقدار
<span dir="ltr">220 bpm</span>
بودند.
</p>

In [ ]:
heart_rate = physiological_df["Heart Rate (bpm)"]
q1 = heart_rate.quantile(0.25)
q3 = heart_rate.quantile(0.75)

iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

heart_rate_outliers = physiological_df[
    (heart_rate < lower_bound) |
    (heart_rate > upper_bound)
]

print("Q1:", q1)
print("Q3:", q3)
print("IQR:", iqr)
print("Lower bound:", lower_bound)
print("Upper bound:", upper_bound)
print("Number of outliers:", len(heart_rate_outliers))

print(
    heart_rate_outliers["Heart Rate (bpm)"]
    .value_counts()
    .sort_index()
)

<h4 dir="rtl" style="text-align: right;">
بررسی ویژگی‌های مرتبط با داده‌های پرت ضربان قلب
</h4>

<p dir="rtl" style="text-align: right;">
پیش از اصلاح داده‌های پرت، ردیف‌های دارای ضربان قلب غیرعادی در کنار تعدادی از متغیرهای مرتبط بررسی شدند. هدف از این بررسی، تشخیص وجود یک الگوی منطقی یا فیزیولوژیک برای مقادیر
<span dir="ltr">220 bpm</span>
بود.
</p>

<p dir="rtl" style="text-align: right;">
برای این منظور، شاخص ردیف‌های پرت استخراج شد و متغیرهایی مانند سن، نرخ تنفس، میزان تعریق، سرگیجه، استرس، اضطراب و مصرف دارو در همان ردیف‌ها مورد بررسی قرار گرفتند.
</p>

<p dir="rtl" style="text-align: right;">
بررسی این ردیف‌ها الگوی مشخص و یکسانی را در سایر متغیرهای مرتبط نشان نداد. همچنین تکرار دقیق مقدار
<span dir="ltr">220</span>
در تمام داده‌های پرت، احتمال وجود خطای ثبت یا مقدار غیرعادی سیستماتیک را تقویت می‌کند.
</p>

In [ ]:
outlier_indices = heart_rate_outliers.index

outlier_context = df.loc[
    outlier_indices,
    [
        "Age",
        "Heart Rate (bpm)",
        "Breathing Rate (breaths/min)",
        "Sweating Level (1-5)",
        "Dizziness",
        "Stress Level (1-10)",
        "Anxiety Level (1-10)",
        "Medication"
    ]
]

outlier_context

<h4 dir="rtl" style="text-align: right;">
اصلاح داده‌های پرت ضربان قلب با روش Clip
</h4>

<p dir="rtl" style="text-align: right;">
پس از شناسایی داده‌های پرت، برای کاهش اثر آن‌ها از روش
<span dir="ltr">Clipping</span>
استفاده شد. در این روش، مقادیر کمتر از کران پایین با کران پایین و مقادیر بیشتر از کران بالا با کران بالا جایگزین می‌شوند.
</p>

<p dir="rtl" style="text-align: right;">
برای حفظ داده‌های اولیه، مقادیر اصلاح‌شده در ستون جدید
<span dir="ltr">Heart Rate Cleaned</span>
ذخیره شدند و ستون اصلی بدون تغییر باقی ماند. در نتیجه، مقادیر
<span dir="ltr">220 bpm</span>
با کران بالای محاسبه‌شده، یعنی
<span dir="ltr">153.5 bpm</span>
جایگزین شدند.
</p>

In [ ]:
physiological_df["Heart Rate (bpm)"] = heart_rate.clip(
    lower=lower_bound,
    upper=round(upper_bound)
)

<h4 dir="rtl" style="text-align: right;">
بررسی داده‌های پرت نرخ تنفس
</h4>

<p dir="rtl" style="text-align: right;">
برای بررسی وجود داده‌های پرت در ستون
<span dir="ltr">Breathing Rate</span>
نیز از روش
<span dir="ltr">IQR</span>
استفاده شد. ابتدا چارک اول، چارک سوم و فاصله میان‌چارکی محاسبه شدند و سپس کران‌های پایین و بالا به دست آمدند.
</p>

<p dir="rtl" style="text-align: right;">
کران پایین این ستون برابر با
<span dir="ltr">3.5</span>
و کران بالا برابر با
<span dir="ltr">39.5</span>
محاسبه شد. هیچ مقداری خارج از این محدوده قرار نداشت؛ بنابراین، داده پرت آماری در ستون نرخ تنفس شناسایی نشد و نیازی به اصلاح مقادیر این ستون وجود ندارد.
</p>

In [ ]:
breathing_rate = physiological_df["Breathing Rate (breaths/min)"]

q1_breathing = breathing_rate.quantile(0.25)
q3_breathing = breathing_rate.quantile(0.75)

iqr_breathing = q3_breathing - q1_breathing

lower_bound_breathing = q1_breathing - 1.5 * iqr_breathing
upper_bound_breathing = q3_breathing + 1.5 * iqr_breathing

breathing_outlier_mask = (
    (breathing_rate < lower_bound_breathing) |
    (breathing_rate > upper_bound_breathing)
)

breathing_rate_outliers = physiological_df[breathing_outlier_mask]

print("Q1:", q1_breathing)
print("Q3:", q3_breathing)
print("IQR:", iqr_breathing)
print("Lower bound:", lower_bound_breathing)
print("Upper bound:", upper_bound_breathing)
print("Number of outliers:", breathing_outlier_mask.sum())

print(
    breathing_rate_outliers["Breathing Rate (breaths/min)"]
    .value_counts()
    .sort_index()
)

<h4 dir="rtl" style="text-align: right;">
بررسی دامنه و فراوانی سطح تعریق
</h4>

<p dir="rtl" style="text-align: right;">
متغیر
<span dir="ltr">Sweating Level</span>
یک متغیر ترتیبی با دامنه معتبر ۱ تا ۵ است. به همین دلیل، به‌جای استفاده از روش‌های تشخیص داده پرت مانند
<span dir="ltr">IQR</span>،
مقادیر یکتا و فراوانی هر سطح بررسی شدند.
</p>

<p dir="rtl" style="text-align: right;">
نتایج نشان داد که این ستون فقط شامل مقادیر ۱، ۲، ۳، ۴ و ۵ است و هیچ مقدار گمشده یا خارج از دامنه‌ای ندارد. همچنین فراوانی سطوح مختلف نسبتاً متعادل است؛ بنابراین، نیازی به اصلاح این ستون وجود ندارد.
</p>

In [ ]:
sweating_level = physiological_df["Sweating Level (1-5)"]

print(sweating_level.unique())

print(
    sweating_level
    .value_counts(dropna=False)
    .sort_index()
)

<h4 dir="rtl" style="text-align: right;">
بررسی مقادیر متغیر سرگیجه
</h4>

<p dir="rtl" style="text-align: right;">
متغیر
<span dir="ltr">Dizziness</span>
یک متغیر دسته‌ای دودویی است. برای بررسی صحت دسته‌بندی این ستون، ابتدا مقادیر یکتای آن و سپس فراوانی هر دسته، همراه با مقادیر گمشده، بررسی شدند.
</p>

<p dir="rtl" style="text-align: right;">
نتایج نشان داد که این ستون فقط شامل دو مقدار معتبر
<span dir="ltr">Yes</span>
و
<span dir="ltr">No</span>
است و هیچ مقدار گمشده یا دسته نامعتبر در آن وجود ندارد. بنابراین، نیازی به اصلاح مقادیر این ستون وجود ندارد.
</p>

In [ ]:
dizziness = physiological_df["Dizziness"]

print(dizziness.unique())
print(dizziness.value_counts(dropna=False))

<h4 dir="rtl" style="text-align: right;">
جمع‌بندی پاک‌سازی متغیرهای فیزیولوژیک
</h4>

<p dir="rtl" style="text-align: right;">
در این بخش، نوع داده، مقادیر گمشده، دامنه مقادیر و داده‌های پرت چهار متغیر فیزیولوژیک بررسی شدند.
</p>

<p dir="rtl" style="text-align: right;">
در ستون
<span dir="ltr">Heart Rate</span>
تعداد ۳۰ مقدار پرت برابر با
<span dir="ltr">220 bpm</span>
شناسایی شد. این مقادیر با استفاده از روش
<span dir="ltr">Clipping</span>
و بر اساس کران بالای
<span dir="ltr">IQR</span>
اصلاح و در ستون جدید
<span dir="ltr">Heart Rate Cleaned</span>
ذخیره شدند.
</p>

<p dir="rtl" style="text-align: right;">
در ستون
<span dir="ltr">Breathing Rate</span>
هیچ داده پرتی شناسایی نشد. ستون
<span dir="ltr">Sweating Level</span>
فقط شامل مقادیر معتبر ۱ تا ۵ و ستون
<span dir="ltr">Dizziness</span>
فقط شامل دسته‌های معتبر
<span dir="ltr">Yes</span>
و
<span dir="ltr">No</span>
بودند. بنابراین، این سه ستون بدون تغییر باقی ماندند.
</p>

<p dir="rtl" style="text-align: right;">
همچنین هیچ مقدار گمشده یا غیرقابل‌تبدیل به عدد در متغیرهای فیزیولوژیک مشاهده نشد.
</p>

<h3 dir="rtl" style="text-align: right;">
3.4 درمان، سابقه و متغیر هدف:<br>

<span dir="ltr">Family History, Medication, Therapy Sessions,</span><br>

<span dir="ltr">Therapy History, Recent Major Life Event, Stress Level, Anxiety Level, Target</span>
</h3>

In [ ]:
treatment_cols = ['Family History of Anxiety', 'Medication', 'Therapy Sessions (per month)', 'Therapy History',
                  'Recent Major Life Event', 'Stress Level (1-10)', 'Anxiety Level (1-10)', 'Target', 'is_Anxious']

df[treatment_cols].describe(include='all').T

In [ ]:
# آیا Target و is_Anxious یکی هستند؟
same_target_mask = df['Target'] == df['is_Anxious']
print(f'Target == is_Anxious در {same_target_mask.sum()} سطر از {df.shape[0]}')

# Target بررسی
print()
print(df.groupby('Target')['Anxiety Level (1-10)'].agg(['min', 'max', 'count']))

stress_out_of_scale_mask = df['Stress Level (1-10)'] > 10

# استرس خارج از مقیاس
print()
print(f'استرس بالای 10: {stress_out_of_scale_mask.sum()} سطر، مقدار: {df.loc[stress_out_of_scale_mask, "Stress Level (1-10)"].unique()}')


# Therapy History
print()
print(df['Therapy History'].value_counts(dropna=False))

<h2 dir="rtl" style="text-align: right;">
    مشاهدات
</h2>

<p dir="rtl" style="text-align: right;">
    <strong>is_Anxious</strong> با <strong>Target</strong> کاملاً یکسان است و به نظر می‌رسد
    <strong>is_Anxious</strong> یک کپی از <strong>Target</strong> باشد.
</p>

<h3 dir="rtl" style="text-align: right;">
    تصمیم
</h3>

<p dir="rtl" style="text-align: right;">
    • <strong>Anxiety Level</strong> متغیر اصلی است و <strong>Target</strong> فقط برای رنگ نمودارها و آزمون‌های دسته‌ای استفاده می‌شود.
</p>

<p dir="rtl" style="text-align: right;">
    • <strong>is_Anxious</strong> به دلیل تکراری بودن حذف می‌شود.
</p>

<p dir="rtl" style="text-align: right;">
    • <strong>Therapy History</strong> با ۱۸۲۷ مقدار گمشده از ۲۰۳۰ حذف می‌شود؛ با تنها حدود ۱۰٪ داده، نتیجه قابل اتکایی حاصل نمی‌شود.
</p>

<p dir="rtl" style="text-align: right;">
    • <strong>Stress Level = 15</strong> در ۳۰ سطر → <code>NaN</code> با ثبت فلگ → جایگزینی با <strong>میانه</strong>.
</p>

<p dir="rtl" style="text-align: right;">
    • مقادیر گمشده در <strong>Medication</strong> → <code>Unknown</code>.
</p>

In [ ]:
# فلگ و اصلاح استرس
df['stress_invalid_flag'] = stress_out_of_scale_mask.astype(int)
df.loc[stress_out_of_scale_mask, 'Stress Level (1-10)'] = np.nan

median_stress = df['Stress Level (1-10)'].median()
df['Stress Level (1-10)'] = df['Stress Level (1-10)'].fillna(median_stress)

#Medication گمشده حایگزین با Unknown
df['Medication'] = df['Medication'].fillna('Unknown')

# حذف ستون های  'is_Anxious', 'Therapy History'
df = df.drop(columns=['is_Anxious', 'Therapy History'])

print(f'میانه استرس: {median_stress}')
print(f'ستون های باقی مانده: {df.shape[1]}')

<h3 dir="rtl" style="text-align: right;">
3.5 ادغام پاک سازی ها و ساخت
<span dir="ltr">clean_df</span>
</h3>

<h2 dir="rtl" style="text-align: right;">
4. ویژوال تک متغیره
</h2>

<h3 dir="rtl" style="text-align: right;">
4.1 جمعیت شناختی
</h3>

<h3 dir="rtl" style="text-align: right;">
4.2 سبک زندگی
</h3>

In [ ]:
# =================================
# plot Numeric columns before-after
# =================================
fig, axes = plt.subplots(4, 2, figsize=(14, 22))

cols = [
    'Sleep Hours',
    'Physical Activity (hrs/week)',
    'Caffeine Intake (mg/day)',
    'Alcohol Consumption (drinks/week)'
]

titles = [
    'Sleep Hours',
    'Physical Activity',
    'Caffeine Intake',
    'Alcohol Consumption'
]

bins_list = [20, 20, 20, 10]

for i, col in enumerate(cols):
    
# =========================
# 1. df Before
# =========================
    data_before = df[col].dropna()
    axes[i, 0].hist(
        data_before,
        bins=bins_list[i],
        color='cornflowerblue',
        edgecolor='black',
        alpha=0.8
    )
    axes[i, 0].set_title(f'{titles[i]} - Before', fontsize=13, fontweight='bold')
    axes[i, 0].set_xlabel(col, fontsize=11)
    axes[i, 0].set_ylabel('Frequency', fontsize=11)
    axes[i, 0].grid(axis='y', alpha=0.3)
    
# =========================
# 2. df after
# =========================
    data_after = df_clean[col].dropna()
    axes[i, 1].hist(
        data_after,
        bins=bins_list[i],
        color='#FF6D00',
        edgecolor='black',
        alpha=0.8
    )
    axes[i, 1].set_title(f'{titles[i]} - After', fontsize=13, fontweight='bold')
    axes[i, 1].set_xlabel(col, fontsize=11)
    axes[i, 1].set_ylabel('Frequency', fontsize=11)
    axes[i, 1].grid(axis='y', alpha=0.3)

# =========================
# Layout
# =========================
fig.suptitle(
    'Distribution of Numeric Variables: Before vs After Cleaning',
    fontsize=18,
    fontweight='bold',
    y=0.995
)

plt.tight_layout(rect=[0, 0, 1, 0.98])
plt.show()

In [ ]:
# ==================================
# plot Categorical Data before-after
# ==================================
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# =========================
# 1. Smoking - Before
# =========================
smoking_before = df['Smoking'].value_counts()

axes[0, 0].bar(
    smoking_before.index.astype(str),
    smoking_before.values,
    color='cornflowerblue',
    edgecolor='black',
    alpha=0.8
)
axes[0, 0].set_title('Smoking - Before', fontsize=13, fontweight='bold')
axes[0, 0].set_xlabel('Smoking', fontsize=11)
axes[0, 0].set_ylabel('Frequency', fontsize=11)
axes[0, 0].grid(axis='y', alpha=0.3)

# =========================
# 2. Smoking - After
# =========================
smoking_after = df_clean['Smoking'].value_counts()

axes[0, 1].bar(
    smoking_after.index.astype(str),
    smoking_after.values,
    color='#FF6D00',
    edgecolor='black',
    alpha=0.8
)
axes[0, 1].set_title('Smoking - After', fontsize=13, fontweight='bold')
axes[0, 1].set_xlabel('Smoking', fontsize=11)
axes[0, 1].set_ylabel('Frequency', fontsize=11)
axes[0, 1].grid(axis='y', alpha=0.3)

# =========================
# 3. Diet Quality - Before
# =========================
diet_before = df['Diet Quality (1-10)'].value_counts().sort_index()

axes[1, 0].bar(
    diet_before.index.astype(str),
    diet_before.values,
    color='cornflowerblue',
    edgecolor='black',
    alpha=0.8
)
axes[1, 0].set_title('Diet Quality - Before', fontsize=13, fontweight='bold')
axes[1, 0].set_xlabel('Diet Quality (1-10)', fontsize=11)
axes[1, 0].set_ylabel('Frequency', fontsize=11)
axes[1, 0].grid(axis='y', alpha=0.3)

# =========================
# 4. Diet Quality - After
# =========================
diet_after = df_clean['Diet Quality (1-10)'].value_counts().sort_index()

axes[1, 1].bar(
    diet_after.index.astype(str),
    diet_after.values,
    color='#FF6D00',
    edgecolor='black',
    alpha=0.8
)
axes[1, 1].set_title('Diet Quality - After', fontsize=13, fontweight='bold')
axes[1, 1].set_xlabel('Diet Quality (1-10)', fontsize=11)
axes[1, 1].set_ylabel('Frequency', fontsize=11)
axes[1, 1].grid(axis='y', alpha=0.3)

# =========================
# Layout
# =========================
fig.suptitle(
    'Distribution of Categorical and Discrete Variables: Before vs After',
    fontsize=16,
    fontweight='bold',
    y=0.98
)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

<h3 dir="rtl" style="text-align: right;">
4.3 فیزیولوژیک
</h3>

<h3 dir="rtl" style="text-align: right;">
4.4 درمان، سابقه و متغیر هدف
</h3>

<h2 dir="rtl" style="text-align: right;">
5. ویژوال دومتغیره
</h2>

<h3 dir="rtl" style="text-align: right;">
5.1 ستون های دسته ای با اضطراب
</h3>

<h3 dir="rtl" style="text-align: right;">
5.2 ستون های عددی با اضطراب
</h3>

In [ ]:

numeric_cols = [
    'Age',
    'Sleep Hours',
    'Physical Activity (hrs/week)',
    'Caffeine Intake (mg/day)',
    'Alcohol Consumption (drinks/week)',
    'Stress Level (1-10)',
    'Heart Rate (bpm)',
    'Breathing Rate (breaths/min)',
    'Sweating Level (1-5)',
    'Therapy Sessions (per month)',
    'Diet Quality (1-10)'
]

fig, axes = plt.subplots(6, 2, figsize=(11, 18))
axes = axes.flatten()

box_colors = ['#A8D5E5', '#F4A261']   

for i, col in enumerate(numeric_cols):
    bp = df_clean.boxplot(
        column=col,
        by='Target',
        ax=axes[i],
        patch_artist=True,          
        return_type='dict'
    )
    
    for patch, color in zip(bp[col]['boxes'], box_colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.8)
    
    for median in bp[col]['medians']:
        median.set_color('#1a1a1a')
        median.set_linewidth(1.5)

    axes[i].set_title(f'{col} vs Anxiety Status',
                      fontsize=12,
                      fontweight='bold',
                      pad=12)    

    axes[i].set_xlabel('is_Anxious')
    axes[i].set_ylabel(col)
    axes[i].grid(axis='y', alpha=0.3)

for j in range(len(numeric_cols), len(axes)):
    fig.delaxes(axes[j])

# =========================
# Layout
# =========================
plt.suptitle(
    'Bivariate Analysis: Numeric Variables vs Anxiety Status',
    fontsize=18,
    fontweight='bold',
    y=0.995           
)

plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.subplots_adjust(hspace=0.45)   
plt.show()

<h2 dir="rtl" style="text-align: right;">
6. آزمون های آماری
</h2>

<h3 dir="rtl" style="text-align: right;">
6.1 آزمون همبستگی: عددی با عددی
</h3>

<h3 dir="rtl" style="text-align: right;">
6.2 آزمون
<span dir="ltr">t-test</span> و
<span dir="ltr">ANOVA</span>:
دسته ای با عددی
</h3>

<h3 dir="rtl" style="text-align: right;">
6.3 آزمون
<span dir="ltr">chi-square</span>:
دسته ای با دسته ای
</h3>

<h3 dir="rtl" style="text-align: right;">
6.4 امتیازی: بازه های اطمینان
</h3>

<h2 dir="rtl" style="text-align: right;">
7.
<span dir="ltr">KPI</span>
و فیچرهای تعاملی
</h2>

<h3 dir="rtl" style="text-align: right;">
7.1 استخراج
<span dir="ltr">KPI</span>
و فیچرهای تعاملی از ستون ها
</h3>

<h3 dir="rtl" style="text-align: right;">
7.2 ویژوال
<span dir="ltr">KPI</span>
و فیچرهای جدید
</h3>

<h2 dir="rtl" style="text-align: right;">
8. ویژوال چندمتغیره: ترکیب ویژگی ها و گروه های پرخطر
</h2>

In [ ]:
plot_df = df_clean[['Stress Level (1-10)', 'Sleep Hours', 'Target']]

plt.figure(figsize=(10, 7))

sns.scatterplot(
    data=plot_df,
    x='Stress Level (1-10)',
    y='Sleep Hours',
    hue='Target',
    palette={0: '#2196F3', 1: '#FF6D00'},   
    s=90,
    alpha=0.8,
    edgecolor='white',
    linewidth=0.7
)

plt.title('Stress Level vs Sleep Hours by Anxiety Status', 
          fontsize=16, fontweight='bold', pad=18)

plt.xlabel('Stress Level (1-10)', fontsize=12)
plt.ylabel('Sleep Hours', fontsize=12)

plt.legend(title='is_Anxious', title_fontsize=12, fontsize=11, 
           loc='upper right', frameon=True)

plt.grid(True, alpha=0.3, linestyle='--')
plt.tight_layout()
plt.show()

print(df.)

In [ ]:
plot_df = df_clean[['Physical Activity (hrs/week)', 'Stress Level (1-10)', 'Target']]

plt.figure(figsize=(10, 7))

sns.scatterplot(
    data=plot_df,
    x='Stress Level (1-10)',
    y='Physical Activity (hrs/week)',
    hue='Target',
    palette={0: '#2196F3', 1: '#FF6D00'},   
    s=90,
    alpha=0.8,
    edgecolor='white',
    linewidth=0.7
)

plt.title('Stress Level vs Physical Activity by Anxiety Status', 
          fontsize=16, fontweight='bold', pad=18)

plt.xlabel('Stress Level (1-10)', fontsize=12)
plt.ylabel('Physical Activity (hrs/week)', fontsize=12)

plt.legend(title='is_Anxious', title_fontsize=12, fontsize=11, 
           loc='upper right', frameon=True)

plt.grid(True, alpha=0.3, linestyle='--')
plt.tight_layout()
plt.show()

In [ ]:
plot_df = df_clean[['Heart Rate (bpm)', 'Breathing Rate (breaths/min)', 'Target']]

plt.figure(figsize=(10, 7))

sns.scatterplot(
    data=plot_df,
    x='Heart Rate (bpm)',
    y='Breathing Rate (breaths/min)',
    hue='Target',
    palette={0: '#2196F3', 1: '#FF6D00'},   
    s=90,
    alpha=0.8,
    edgecolor='white',
    linewidth=0.7
)

plt.title('Heart Rate vs Breathing Rate by Anxiety Status', 
          fontsize=16, fontweight='bold', pad=18)

plt.xlabel('Heart Rate (bpm)', fontsize=12)
plt.ylabel('Breathing Rate (breaths/min)', fontsize=12)

plt.legend(title='is_Anxious', title_fontsize=12, fontsize=11, 
           loc='upper right', frameon=True)

plt.grid(True, alpha=0.3, linestyle='--')
plt.tight_layout()
plt.show()

In [ ]:
plot_df = df_clean[['Age', 'Gender', 'Target']]

plt.figure(figsize=(10, 7))

sns.boxplot(
    data=plot_df,
    x='Gender',
    y='Age',
    hue='Target',
    palette={0: '#2196F3', 1: '#FF6D00'},
    width=0.6,
    linewidth=1.3
)

plt.title('Age Distribution by Gender and Anxiety Status', 
          fontsize=16, fontweight='bold', pad=18)

plt.xlabel('Gender', fontsize=12)
plt.ylabel('Age', fontsize=12)

plt.legend(title='is_Anxious', title_fontsize=12, fontsize=11, 
           loc='upper right', frameon=True)

plt.grid(axis='y', alpha=0.3, linestyle='--')
plt.tight_layout()
plt.show()

In [ ]:
plot_df = df_clean[['Caffeine Intake (mg/day)', 'Sleep Hours', 'Target']]

plt.figure(figsize=(10, 7))

sns.scatterplot(
    data=plot_df,
    x='Caffeine Intake (mg/day)',
    y='Sleep Hours',
    hue='Target',
    palette={0: '#2196F3', 1: '#FF6D00'},   
    s=90,
    alpha=0.8,
    edgecolor='white',
    linewidth=0.7
)

plt.title('Caffeine Intake vs Sleep Hours by Anxiety Status', 
          fontsize=16, fontweight='bold', pad=18)

plt.xlabel('Caffeine Intake (mg/day)', fontsize=12)
plt.ylabel('Sleep Hours', fontsize=12)

plt.legend(title='is_Anxious', title_fontsize=12, fontsize=11, 
           loc='upper right', frameon=True)

plt.grid(True, alpha=0.3, linestyle='--')
plt.tight_layout()
plt.show()

In [ ]:
plot_df = df_clean[['Diet Quality (1-10)', 'Physical Activity (hrs/week)', 'Target']]

plt.figure(figsize=(10, 7))

sns.scatterplot(
    data=plot_df,
    x='Diet Quality (1-10)',
    y='Physical Activity (hrs/week)',
    hue='Target',
    palette={0: '#2196F3', 1: '#FF6D00'},   
    s=90,
    alpha=0.8,
    edgecolor='white',
    linewidth=0.7
)

plt.title('Diet Quality vs Physical Activity by Anxiety Status', 
          fontsize=16, fontweight='bold', pad=18)

plt.xlabel('Diet Quality (1-10)', fontsize=12)
plt.ylabel('Physical Activity (hrs/week)', fontsize=12)

plt.legend(title='is_Anxious', title_fontsize=12, fontsize=11, 
           loc='upper right', frameon=True)

plt.grid(True, alpha=0.3, linestyle='--')
plt.tight_layout()
plt.show()

<h2 dir="rtl" style="text-align: right;">
9. امتیازی: ویژوال سه متغیره و بیشتر
</h2>